In [1]:
# imports
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

# linear models
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingClassifier

# model tuning and preprocessing
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler,FunctionTransformer,OneHotEncoder
from sklearn.compose import ColumnTransformer

# model metrics
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import accuracy_score, recall_score
from sklearn.metrics import classification_report
import time

## Descriptors

In [3]:
# import pandas as pd
# import io

# def reformat_amino_acid_descriptors(file_name, output_file):
#     """Reads the concatenated descriptor file, merges blocks, and saves a clean CSV."""
    
#     # --- Step 1: Read the raw content ---
#     # Reading the whole content as a string first to handle the non-standard block structure
#     try:
#         with open(file_name, 'r') as f:
#             csv_content = f.read()
#     except FileNotFoundError:
#         # Handling the absolute path issue by using the accessible file name
#         with open("merge-csv.com__691ca36543fdc.csv", 'r') as f:
#             csv_content = f.read()

#     # Split the content into blocks based on the AA_3 header row, which marks a new table
#     lines = [line.strip() for line in csv_content.split('\n') if not line.startswith('#') and line.strip()]
#     content_clean = '\n'.join(lines)
#     # The split key must be a pattern that separates the blocks
#     blocks = content_clean.split('AA_3,AA_1')

#     main_df = None

#     # --- Step 2: Iterate and Merge Blocks ---
#     for i, block in enumerate(blocks):
        # if not block.strip():
        #     continue

        # # Reconstruct the block content for pd.read_csv
        # if i > 0:
        #     block_content = 'AA_3,AA_1' + block.split('AA_3,AA_1')[-1]
        # else:
        #     # Handle the first block's messy headers
        #     block_lines = block.strip().split('\n')
        #     start_index = next((j for j, line in enumerate(block_lines) if line.startswith('AA_3,AA_1')), None)
            
        #     if start_index is None:
        #         continue

        #     header_line = block_lines[start_index]
        #     data_lines = block_lines[start_index+1:]
        #     block_content = '\n'.join([header_line] + data_lines)

        # # Read the block. Using the 'python' engine and skipping bad lines for robustness
        # temp_df = pd.read_csv(
        #     io.StringIO(block_content), 
        #     engine='python',
        #     on_bad_lines='skip' # Crucial fix to bypass single corrupted lines
        # )

        # # Cleanup: Drop redundant columns and use AA_1_Code as the key
        # temp_df = temp_df.drop(columns=['Name', 'AA_3'], errors='ignore').set_index('AA_1')

        # if main_df is None:
        #     main_df = temp_df
        # else:
        #     # Merge on the 1-letter code 'AA_1'
        #     main_df = main_df.merge(temp_df, left_index=True, right_index=True, how='inner')

#     # --- Step 3: Final Cleanup and Save ---
#     final_df = main_df.reset_index().rename(columns={'AA_1': 'AA_1_Code'})
#     # Clean up column names
#     final_df.columns = final_df.columns.str.replace(r'[^a-zA-Z0-9_]', '', regex=True)
#     final_df = final_df.rename(columns={'Ist': 'Ist_PCA', '2nd': '2nd_PCA', '3rd': '3rd_PCA'})
    
#     # Save to a new file
#     final_df.to_csv(output_file, index=False)
    
#     return final_df

# # 1. Clean the original file and save it as a new CSV
# clean_file_name = "clean_amino_acids.csv"
# full_content = reformat_amino_acid_descriptors("C:/Users/Trish/Downloads/merge-csv.com__691ca36543fdc.csv", clean_file_name)

# # 2. Now we can read the clean file successfully
# df = pd.read_csv(clean_file_name)

# print("DataFrame loaded successfully after reformatting:")
# print(df.head())
# print(df.info())

## Actually Data Modeling

In [2]:
descriptors = pd.read_csv("clean_amino_acids.csv")

In [3]:
test_X = pd.read_csv("~/Documents/CMU/Computational Modeling/hw-4-classification-of-green-fluorescent-protein/test_X.csv")
train_X = pd.read_csv("~/Documents/CMU/Computational Modeling/hw-4-classification-of-green-fluorescent-protein/train_X.csv", index_col = "Unnamed: 0")
train_y = pd.read_csv("~/Documents/CMU/Computational Modeling/hw-4-classification-of-green-fluorescent-protein/train_Y.csv", index_col = "Unnamed: 0")
test_y = pd.read_csv("~/Documents/CMU/Computational Modeling/hw-4-classification-of-green-fluorescent-protein/y_sample_submission.csv")

In [4]:
train_X

,ConstructedAASeq_cln,Id
0,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,11328
1,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,5781
2,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,13681
3,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,30804
4,SKGEELFTGVVPILVELDGDVNGHTFSVSGEGEGDATYGELTLKFI...,30813
...,...,...
33024,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,7024
33025,SKGEELFTGVVPTLVELDGDVNGHKFSVSGEGAGDATYSKLTLKFI...,14012
33026,SKGEELFTGVVPVLVELDGDVNGHKFSVSGEGEGDATYGKLTLKLI...,4140
33027,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,15193


In [7]:
train_y = train_y.drop('Id', axis=1)

## Hist Gradient Boosting

In [11]:
df_aa_props = descriptors.set_index('AA_1_Code')
aa_map = df_aa_props.T.to_dict('list')
property_cols = df_aa_props.columns.tolist()

def get_sequence_mean_properties(sequence, aa_map, property_cols):
    sequence = str(sequence)
    if not sequence:
        return pd.Series([np.nan] * len(property_cols), index=[f'mean_{col}' for col in property_cols])
    property_vectors = [
        aa_map.get(aa, [np.nan] * len(property_cols))
        for aa in sequence
    ]
    property_matrix = np.array(property_vectors, dtype=float)

    mean_properties = np.nanmean(property_matrix, axis=0)

    return pd.Series(mean_properties, index=[f'mean_{col}' for col in property_cols])
    
new_features = train_X['ConstructedAASeq_cln'].apply(
    lambda x: get_sequence_mean_properties(x, aa_map, property_cols)
)

# Combine the original DataFrame with the new features
big_X = pd.concat([train_X, new_features], axis=1)
big_X = pd.DataFrame(big_X).reset_index()

In [12]:
train_y["Brightness_Class"].value_counts(normalize=True)

Brightness_Class
0    0.610655
1    0.389345
Name: proportion, dtype: float64

In [13]:
def split_sequence_column(X):
    sequence_data = X[:, 1] 
    sequence_series = pd.Series(sequence_data)
    return sequence_series.apply(lambda s: list(s)).tolist()


splitter = FunctionTransformer(split_sequence_column, validate=False)

onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

pipe = Pipeline([
    ("split", splitter),
    ("onehot", onehot),
    ("clf", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.1))
])

In [27]:
train_y_single = train_y.iloc[:, 0].to_numpy()

big_X_numpy = big_X.to_numpy()

pipe.fit(big_X_numpy, train_y_single)

,steps,"[('split', ...), ('onehot', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function spl...00299B25365C0>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,None
,kw_args,None


In [28]:
param_grid = {
    'clf__max_depth': [6, 8, 10],  
    'clf__learning_rate': [0.1, 0.05], 
    'clf__max_leaf_nodes': [20, 31],
}

print("Starting Grid Search...")
start_time = time.time()

grid_search = GridSearchCV(
    estimator=pipe, 
    param_grid=param_grid, 
    cv=3, 
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(big_X_numpy, train_y_single)

end_time = time.time()
print(f"Grid Search completed in {end_time - start_time:.2f} seconds.")

best_model = grid_search.best_estimator_
print("\nBest parameters found: ", grid_search.best_params_)

y_predict = best_model.predict(big_X_numpy)
y_true = train_y_single

print("\n--- Classification Report (y_true vs y_predict) ---")
print(classification_report(y_true, y_predict))

Starting Grid Search...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Grid Search completed in 512.10 seconds.

Best parameters found:  {'clf__learning_rate': 0.1, 'clf__max_depth': 10, 'clf__max_leaf_nodes': 20}

--- Classification Report (y_true vs y_predict) ---
              precision    recall  f1-score   support

           0       0.93      0.82      0.87     18948
           1       0.76      0.90      0.83     12081

    accuracy                           0.85     31029
   macro avg       0.85      0.86      0.85     31029
weighted avg       0.86      0.85      0.85     31029



In [31]:
test_X = test_X[test_X['Id'].isin(test_y['Id'])]
test_X

,ConstructedAASeq_cln,Id
0,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,50579
1,SKGEELFTGVVPILVELDGDVSGHKFSVSGEGEGDATYGKLTLKFI...,37987
2,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,53977
3,SKGEELFTGVVPILVELDGDVNGHKLSVSGEGEGDATYGKLTLKFI...,10677
4,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,35653
...,...,...
20681,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,35750
20682,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,53940
20683,SKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFI...,14948
20684,SKGEELFTRVVPILVGLDGDVNGHKFSVSGEGEGDATYGKLTQKFI...,17040


In [33]:
test_new_features = test_X['ConstructedAASeq_cln'].apply(
    lambda x: get_sequence_mean_properties(x, aa_map, property_cols)
)

big_test_X = pd.concat([test_X, test_new_features], axis=1)

big_test_X_numpy = big_test_X.to_numpy()

test_y_single = test_y.drop('Id', axis=1).iloc[:, 0].to_numpy()

In [35]:
test_new_features = test_X['ConstructedAASeq_cln'].apply(
    lambda x: get_sequence_mean_properties(x, aa_map, property_cols)
)
big_test_X_df = pd.concat([test_X.reset_index(), test_new_features], axis=1)

big_test_X_df = big_test_X_df.drop(columns=['index', 'Id'])

test_X_for_merge = test_X.copy().reset_index() 
big_test_X_df = pd.concat([test_X_for_merge, test_new_features], axis=1) 

test_X_reset = test_X.copy().reset_index()
test_new_features = test_X['ConstructedAASeq_cln'].apply(
    lambda x: get_sequence_mean_properties(x, aa_map, property_cols)
)
big_test_X_df = pd.concat([test_X_reset, test_new_features], axis=1)

big_test_X_df = big_test_X_df.drop(columns=['Id'])

big_test_X_numpy = big_test_X_df.to_numpy()

test_y_single = test_y.drop('Id', axis=1).iloc[:, 0].to_numpy() 

best_model = grid_search.best_estimator_

y_predict = best_model.predict(big_test_X_numpy)

y_true = test_y_single

print("\n--- Classification Report on UNSEEN Test Data ---")
print(classification_report(y_true, y_predict))


--- Classification Report on UNSEEN Test Data ---
              precision    recall  f1-score   support

           0       0.61      0.54      0.57     12548
           1       0.40      0.46      0.43      8138

    accuracy                           0.51     20686
   macro avg       0.50      0.50      0.50     20686
weighted avg       0.52      0.51      0.51     20686



In [ ]:
submission = pd.DataFrame({'Id': test_X['Id'],'Brightness_Class': y_predict})
submission.to_csv("submission.csv", index=False)

## Logistic Regression

In [61]:
def split_sequence_column(X):
    sequence_series = pd.Series(X[:, 0]) 
    return sequence_series.apply(lambda s: list(s)).tolist()

splitter = FunctionTransformer(split_sequence_column, validate=False)
onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

train_y_single = train_y.iloc[:, 0].to_numpy()
big_X_numpy = big_X.to_numpy()

big_test_X = big_test_X.reindex(columns=big_X.columns, fill_value=0)
big_test_X_numpy = big_test_X.to_numpy()

y_true = test_y['Brightness_Class'].to_numpy()

sequence_pipeline = Pipeline([
    ("split", splitter), 
    ("onehot", onehot)
])

preprocessor = ColumnTransformer(
    transformers=[
        ('sequence', sequence_pipeline, [1]), 
        ('scale', StandardScaler(), slice(3, big_X_numpy.shape[1]))
    ],
    remainder='drop'
)

pipeline = Pipeline([
    ("preprocessor", preprocessor), 
    ("model", LogisticRegression(max_iter=10000, solver='liblinear')),
])

param_grid = [{
    "model__penalty": ["l1", "l2"],
    "model__C": 1 / (10.0 ** np.arange(-4, 3)),
}]

grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring=["accuracy", "recall"],
    refit="accuracy",
    n_jobs=-1
)

_ = grid_search.fit(big_X_numpy, train_y_single)

print("Best parameters found:", grid_search.best_params_)
print("Best cross-validated accuracy:", grid_search.best_score_)

y_predict = grid_search.predict(big_test_X_numpy)

test_accuracy = accuracy_score(y_true, y_predict)
test_recall = recall_score(y_true, y_predict)

print(f"Test accuracy: {test_accuracy:.2f}")
print(f"Test recall: {test_recall:.2f}")

Best parameters found: {'model__C': 100.0, 'model__penalty': 'l2'}
Best cross-validated accuracy: 0.8947436119499649
Test accuracy: 0.52
Test recall: 0.42


In [63]:
print(classification_report(y_true, y_predict))

              precision    recall  f1-score   support

           0       0.61      0.59      0.60     12548
           1       0.40      0.42      0.41      8138

    accuracy                           0.52     20686
   macro avg       0.50      0.50      0.50     20686
weighted avg       0.52      0.52      0.52     20686



In [65]:
submission2 = pd.DataFrame({'Id': test_X['Id'],'Brightness_Class': y_predict})
submission2.to_csv("submission_2.csv", index=False)

## Write-up:

  The objective of this assignment is to classify if a mutant protein sequence would be Bright or not. To achieve this, I examined 2 methods of classification: Logistic regression and hist gradient boost. Before classification I needed to fix the test and training features of the dataset because they only contained the Id of the strain and its encoded seqwuence. However, there was a folder called descriptors that contained different atributes of the amino acids present. What I did then was combined all the csv's in the descriptors folder, then line by line I took the mean of all the properties of the encoded secuence of amino acids.
  Once that was done I also needed to match the number of text_X observations to the sample test_y submission that was available so I could use tgose as my training and testing data, respectively.
  The first model I tested was the hist gradient boot model. The accuracy was approximately 0.5 so the model does as well as random guessing. Thus I needed a more robust model. So then I tried Logistic regression with loss terms. In the end this model's F1 score and accuaracy was a slightly better compared to the HistGradient model, it still performed as well as the previous model.

  I do think there is a model that contains the right capabilities to handle the data and noise. Otherise, I can continue tweaking the logistic regression.